<a href="https://colab.research.google.com/github/nikhilhyperneuron/Qupus-tts/blob/main/NewTTSResearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
HF_TOKEN =userdata.get('HF_TOKEN')


from huggingface_hub import snapshot_download
snapshot_download("Nikya/qwen-snac-tts-finetuning",token=HF_TOKEN,local_dir="qwen-snac-tts")

from huggingface_hub import snapshot_download
snapshot_download("Nikya/qupus_tts_0.8Bsbase",token=HF_TOKEN,local_dir="qupus_tts_0.8Bsbase")
from huggingface_hub import snapshot_download
snapshot_download("Nikya/qupus_tts_0.8Bs",token=HF_TOKEN,local_dir="qupus_tts_0.8B")

Fetching 93 files:   0%|          | 0/93 [00:00<?, ?it/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

'/content/qupus_tts_0.8B'

In [11]:
from datasets import load_dataset
from google.colab import userdata
HF_TOKEN =userdata.get('HF_TOKEN')

"""Assamese	Female	29.09	15,084
Assamese	Male	29.23	16,614
Bengali	Female	29.69	15,570
Bengali	Male	27.06	15,639
Bodo	Female	27.32	16,329
Bodo	Male	24.99	13,162
Dogri	Female	25.69	13,177
Dogri	Male	22.35	10,332
Gujarati	Female	25.39	12,151
Gujarati	Male	26.01	15,799
Hindi	Female	27.05	15,107
Hindi	Male	23.78	13,464
Kannada	Female	27.02	14,913
Kannada	Male	27.59	15,999
Kashmiri	Female	10.95	6,653
Kashmiri	Male	25.34	16,059
Konkani	Female	26.33	17,585
Konkani	Male	25.43	19,324
Maithili	Female	21.79	12,446
Maithili	Male	29.34	12,917
Malayalam	Female	26.42	16,974
Malayalam	Male	25.22	16,877
Manipuri	Female	27.49	13,516
Manipuri	Male	23.55	12,579
Marathi	Female	28.80	15,472
Marathi	Male	26.55	14,483
Nepali	Female	28.74	16,016
Nepali	Male	26.36	15,239
Odia	Female	24.07	11,757
Odia	Male	23.85	13,611
Punjabi	Female	26.72	13,420
Punjabi	Male	29.12	15,620
Sanskrit	Female	25.27	12,757
Sanskrit	Male	26.20	11,002
Santali	Female	27.77	14,469
Santali	Male	25.04	13,513
Sindhi	Female	24.61	13,588
Sindhi	Male	27.58	15,659
Tamil	Female	29.94	19,871
Tamil	Male	26.18	16,790
Telugu	Female	27.28	15,404
Telugu	Male	24.98	15,006
Urdu	Female	26.23	13,980
Urdu	Male	26.03	15,023"""
# 2. Load only the specified parquet shards using streaming
datasetllist = []
for language in ["Hindi","Gujarati","Assamese","Punjabi"]:

  datasetllist.append( load_dataset(
      "ai4bharat/Rasa",
      name=f"{language}",
      split="train",
      streaming=False,
        token=HF_TOKEN
  )
  )

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Gujarati/train-00000-of-00048.parquet:   0%|          | 0.00/293M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [12]:
datasetllist

[Dataset({
     features: ['filename', 'text', 'language', 'gender', 'style', 'duration', 'wav_path', 'audio'],
     num_rows: 25713
 })]

In [13]:
from datasets import concatenate_datasets
dataset = concatenate_datasets(datasetllist)

In [14]:
dataset

Dataset({
    features: ['filename', 'text', 'language', 'gender', 'style', 'duration', 'wav_path', 'audio'],
    num_rows: 25713
})

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install librosa soundfile evaluate jiwer torchcodec "datasets>=3.4.1,<4.0.0"

In [ ]:
!pip install snac torchao --upgrade

In [15]:
from unsloth import FastLanguageModel
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
import torch
import torchaudio
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

from snac import SNAC
from google.colab import userdata
HF_TOKEN =userdata.get('HF_TOKEN')
# ==========================================
# 1. INITIALIZATION & CONFIGURATION
# ==========================================
# Use the *Base* (non-instruct) checkpoint: we repurpose the LM head to emit audio
# tokens, so chat/reasoning alignment is dead weight here.
MODEL_NAME = "/content/qupus_tts_0.8Bsbase"
# MODEL_NAME = "Nikya/qupus_tts_0.8Bs"
MAX_SEQ_LENGTH = 2048             # Adjust based on average audio snippet length
DTYPE = torch.bfloat16            # Set to None for auto-detection
LOAD_IN_4BIT = False               # Use 4bit quantization to drastically reduce memory

print(">>> Loading Unsloth optimized Qwen model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    # max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_16bit=not LOAD_IN_4BIT,
    # device_map="auto",
    token= HF_TOKEN,
    full_finetuning=True
)





🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
>>> Loading Unsloth optimized Qwen model...
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


In [ ]:
# ==========================================
# 2. VOCABULARY EXTENSION FOR SNAC CODES
# ==========================================
if False:
    print(">>> Extending vocabulary with custom SNAC audio tokens...")
    # Control tokens to mark boundaries
    CONTROL_TOKENS = ["<audio_start>", "<audio_end>"]
    tokenizer.add_special_tokens({"additional_special_tokens": CONTROL_TOKENS})

    # Generate unique token names for a 3-layer (24kHz) SNAC setup.
    # snac_24khz has 3 codebooks, each of size 4096 -> 3 * 4096 = 12,288 audio tokens.
    snac_tokens = []
    for layer in range(3):
        for code in range(4096):
            snac_tokens.append(f"<snac_l{layer}_c{code}>")

    # Update tokenizer memory bank
    tokenizer.add_tokens(snac_tokens)
    model.resize_token_embeddings(len(tokenizer))


# ==========================================
# 3. CONSTRUCT PEFT / LORA SPECIFICATION
# ==========================================
print(">>> Attaching LoRA adapters with unconstrained embedding head updates...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
    # CRITICAL: We added thousands of new tokens. We must fully train the
    # embedding matrix and the projection head so they understand them.
    modules_to_save=["embed_tokens", "lm_head"]
)


>>> Attaching LoRA adapters with unconstrained embedding head updates...
Unsloth: Full finetuning is enabled, so .get_peft_model has no effect


In [17]:
dataset

Dataset({
    features: ['filename', 'text', 'language', 'gender', 'style', 'duration', 'wav_path', 'audio'],
    num_rows: 25713
})

In [18]:
# unique_labels = dataset.unique("gender")
# print(unique_labels)  # Output: [1, 0]

['Male', 'Female']


In [19]:
# Define the transformation logic


Map:   0%|          | 0/25713 [00:00<?, ? examples/s]

Dataset({
    features: ['filename', 'text', 'language', 'gender', 'style', 'duration', 'wav_path', 'audio', 'SPEAKER_COLUMN'],
    num_rows: 25713
})

In [24]:

DATASET_NAME = "ai4bharat/Rasa"
DATASET_CONFIG = "Hindi"
DATASET_SPLIT = "train"
AUDIO_COLUMN = "audio"               # HF Audio feature column
TEXT_COLUMN = "text"                   # None -> auto-detect among common names
SPEAKER_COLUMN = "SPEAKER_COLUMN"                # None -> auto-detect; conditions the model on voice identity
TARGET_SR = 24000                    # SNAC 24kHz native rate
MAX_AUDIO_SECONDS = 40               # skip longer clips so SNAC tokens fit in MAX_SEQ_LENGTH

In [ ]:
# Load 24kHz SNAC codec model to process audio into discrete codes
snac_decoder = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().cuda()

def snac_tokens_from_array(audio_array, sr):
    """Flatten an in-memory waveform into Orpheus-style 7-tokens-per-frame strings.

    snac_24khz is hierarchical with strides [1, 2, 4], so the three codebooks
    have different lengths for the same audio:
        codes[0] -> T (coarsest),  codes[1] -> 2T,  codes[2] -> 4T
    The 7-token interleave keeps every code and keeps the layers time-aligned.
    """
    waveform = torch.as_tensor(audio_array, dtype=torch.float32)
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)          # [1, samples]
    elif waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)  # mix to mono

    if sr != TARGET_SR:
        waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)

    waveform = waveform.unsqueeze(0).cuda()       # [batch=1, channels=1, samples]

    with torch.inference_mode():
        codes = snac_decoder.encode(waveform)     # [codes0 (T), codes1 (2T), codes2 (4T)]

    l0 = codes[0][0].cpu().tolist()
    l1 = codes[1][0].cpu().tolist()
    l2 = codes[2][0].cpu().tolist()

    token_sequence = []
    for i in range(len(l0)):
        token_sequence.append(f"<snac_l0_c{l0[i]}>")
        token_sequence.append(f"<snac_l1_c{l1[2 * i]}>")
        token_sequence.append(f"<snac_l2_c{l2[4 * i]}>")
        token_sequence.append(f"<snac_l2_c{l2[4 * i + 1]}>")
        token_sequence.append(f"<snac_l1_c{l1[2 * i + 1]}>")
        token_sequence.append(f"<snac_l2_c{l2[4 * i + 2]}>")
        token_sequence.append(f"<snac_l2_c{l2[4 * i + 3]}>")
    return token_sequence

def build_prompt_prefix(speaker, text_prompt):
    """Orpheus-style multi-speaker prompt: the speaker handle conditions the voice.

    Kept in one helper so training and inference build IDENTICAL prefixes.
    """
    return f"{speaker}: {text_prompt} <audio_start> "

def build_training_example(speaker, text_prompt, audio_tokens):
    """Assemble the prompt + audio token stream with loss masking on the prompt."""
    text_prefix = build_prompt_prefix(speaker, text_prompt)
    audio_suffix = " <audio_end>"

    prefix_ids = tokenizer.encode(text_prefix, add_special_tokens=False)
    audio_ids = tokenizer.encode(" ".join(audio_tokens) + audio_suffix, add_special_tokens=False)

    input_ids = prefix_ids + audio_ids
    labels = [-100] * len(prefix_ids) + audio_ids   # don't compute loss on the text prompt

    if len(input_ids) > MAX_SEQ_LENGTH:
        input_ids = input_ids[:MAX_SEQ_LENGTH]
        labels = labels[:MAX_SEQ_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }

config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/79.5M [00:00<?, ?B/s]

In [ ]:
# # ==========================================
# # 5. LOAD & MAP THE FULL TTS DATASET
# # ==========================================
# from datasets import load_dataset
# from datasets import Audio
# print(f">>> Loading dataset '{DATASET_NAME}' (split='{DATASET_SPLIT}')...")
# raw_ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT)

# # Auto-detect the transcript column if not explicitly set
# if TEXT_COLUMN is None:
#     for cand in ["text_normalized", "text", "transcription", "transcript", "normalized_text", "sentence"]:
#         if cand in raw_ds.column_names:
#             TEXT_COLUMN = cand
#             break
#     if TEXT_COLUMN is None:
#         raise ValueError(f"Could not find a text column in {raw_ds.column_names}. Set TEXT_COLUMN manually.")

# # Auto-detect the speaker column (multi-speaker conditioning)
# if SPEAKER_COLUMN is None:
#     for cand in ["speaker_id", "speaker", "speaker_name", "client_id", "speaker_idx"]:
#         if cand in raw_ds.column_names:
#             SPEAKER_COLUMN = cand
#             break
#     if SPEAKER_COLUMN is None:
#         raise ValueError(f"Could not find a speaker column in {raw_ds.column_names}. Set SPEAKER_COLUMN manually.")
# print(f">>> Columns -> audio='{AUDIO_COLUMN}', text='{TEXT_COLUMN}', speaker='{SPEAKER_COLUMN}'.")
# print(f">>> Distinct speakers in split: {len(set(raw_ds[SPEAKER_COLUMN]))}")

# # Decode + resample audio to 24kHz on access
# raw_ds = raw_ds.cast_column(AUDIO_COLUMN, Audio(sampling_rate=TARGET_SR))

# # Drop clips longer than MAX_AUDIO_SECONDS so we don't truncate speech mid-utterance
# def _within_duration(example):
#     audio = example[AUDIO_COLUMN]
#     return len(audio["array"]) / audio["sampling_rate"] <= MAX_AUDIO_SECONDS

# print(">>> Filtering overly long clips...")
# raw_ds = raw_ds.filter(_within_duration)

# def _encode_example(example):
#     audio = example[AUDIO_COLUMN]
#     tokens = snac_tokens_from_array(audio["array"], audio["sampling_rate"])
#     return build_training_example(example[SPEAKER_COLUMN], example[TEXT_COLUMN], tokens)

# # NOTE: SNAC runs on the GPU, so keep num_proc=1 (single CUDA worker). This is the
# # slow part of the run for large corpora -- it encodes every clip once up front.
# print(">>> Encoding audio -> SNAC tokens and tokenizing (this can take a while)...")
# dataset = raw_ds.map(_encode_example, remove_columns=raw_ds.column_names,num_proc=3)
# print(f">>> Prepared {len(dataset)} training examples.")


In [ ]:
# from datasets import load_dataset
# dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT,streaming=True)
# few_rows = list(dataset.take(20000))
# from datasets import Dataset
# dataset = Dataset.from_list(few_rows)
# dataset

In [32]:
from datasets import load_dataset
from datasets import Audio
from snac import SNAC
import numpy as np

snac_decoder = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().cuda()
ENCODE_BATCH_SIZE = 32               # GPU batch size for SNAC encoding (bigger = faster)
# ENCODED_CACHE_DIR = "/content/qwen-snac-tts/tts_encoded_cache"  # reuse the expensive SNAC encode across runs
ENCODED_CACHE_DIR= "/content/tts_encoded_cache_hindi"
# Calibrate SNAC's time compression (input samples per coarse codes[0] frame) from
# the model itself -- avoids hardcoding a stride that could differ across versions.
with torch.inference_mode():
    _probe = torch.zeros(1, 1, TARGET_SR, device="cuda")
    SNAC_HOP = _probe.shape[-1] / snac_decoder.encode(_probe)[0].shape[-1]
print(f">>> SNAC compression: ~{SNAC_HOP:.1f} samples per coarse frame")

def interleave_snac(l0, l1, l2):
    """Orpheus-style 7-tokens-per-frame flatten of the three SNAC codebooks.

    snac_24khz is hierarchical with strides [1, 2, 4], so for the same audio:
        codes[0] -> T (coarsest),  codes[1] -> 2T,  codes[2] -> 4T
    The 7-token interleave keeps every code and keeps the layers time-aligned.
    """
    token_sequence = []
    for i in range(len(l0)):
        token_sequence.append(f"<snac_l0_c{l0[i]}>")
        token_sequence.append(f"<snac_l1_c{l1[2 * i]}>")
        token_sequence.append(f"<snac_l2_c{l2[4 * i]}>")
        token_sequence.append(f"<snac_l2_c{l2[4 * i + 1]}>")
        token_sequence.append(f"<snac_l1_c{l1[2 * i + 1]}>")
        token_sequence.append(f"<snac_l2_c{l2[4 * i + 2]}>")
        token_sequence.append(f"<snac_l2_c{l2[4 * i + 3]}>")
    return token_sequence

def build_prompt_prefix(speaker, text_prompt):
    """Orpheus-style multi-speaker prompt: the speaker handle conditions the voice.

    Kept in one helper so training and inference build IDENTICAL prefixes.
    """
    return f"{speaker}: {text_prompt} <audio_start> "

def build_training_example(speaker, text_prompt, audio_tokens):
    """Assemble the prompt + audio token stream with loss masking on the prompt."""
    text_prefix = build_prompt_prefix(speaker, text_prompt)
    audio_suffix = " <audio_end>"

    prefix_ids = tokenizer.encode(text_prefix, add_special_tokens=False)
    audio_ids = tokenizer.encode(" ".join(audio_tokens) + audio_suffix, add_special_tokens=False)

    input_ids = prefix_ids + audio_ids
    labels = [-100] * len(prefix_ids) + audio_ids   # don't compute loss on the text prompt

    if len(input_ids) > MAX_SEQ_LENGTH:
        input_ids = input_ids[:MAX_SEQ_LENGTH]
        labels = labels[:MAX_SEQ_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }

def encode_batch(batch):
    """Batched SNAC encode: pad a batch of waveforms, encode once on the GPU, then
    trim each clip's codes back to its true length using SNAC_HOP before flattening.
    """
    get_gpu_memory_map()
    arrays = batch[AUDIO_COLUMN]
    speakers = batch[SPEAKER_COLUMN]
    texts = batch[TEXT_COLUMN]

    # Mono float tensors + true sample lengths (audio is already 24kHz via cast_column)
    waves, lengths = [], []
    for a in arrays:
        w = torch.as_tensor(a["array"], dtype=torch.float32)
        if w.ndim > 1:
            w = w.mean(dim=0)
        waves.append(w)
        lengths.append(w.shape[-1])

    # Right-pad to the longest clip -> [B, 1, max_samples]
    max_len = max(lengths)
    padded = torch.zeros(len(waves), 1, max_len)
    for i, w in enumerate(waves):
        padded[i, 0, :lengths[i]] = w
    padded = padded.cuda()

    with torch.inference_mode():
        codes = snac_decoder.encode(padded)   # [ [B,Tm], [B,2Tm], [B,4Tm] ]
    c0 = codes[0].cpu().tolist()
    c1 = codes[1].cpu().tolist()
    c2 = codes[2].cpu().tolist()
    t_max = len(c0[0])

    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for i in range(len(waves)):
        # Trim padding-induced trailing frames back to this clip's real length
        t0 = max(1, min(round(lengths[i] / SNAC_HOP), t_max))
        l0 = c0[i][:t0]
        l1 = c1[i][:2 * t0]
        l2 = c2[i][:4 * t0]
        tokens = interleave_snac(l0, l1, l2)
        ex = build_training_example(speakers[i], texts[i], tokens)
        out["input_ids"].append(ex["input_ids"])
        out["attention_mask"].append(ex["attention_mask"])
        out["labels"].append(ex["labels"])
    return out

# ==========================================
# 5. LOAD & MAP THE FULL TTS DATASET
# ==========================================
if os.path.isdir(ENCODED_CACHE_DIR):
    from datasets import load_from_disk
    print(f">>> Found cached encoded dataset at '{ENCODED_CACHE_DIR}', loading it...")
    dataset = load_from_disk(ENCODED_CACHE_DIR)
else:
    print(f">>> Loading dataset '{DATASET_NAME}' (split='{DATASET_SPLIT}')...")
    raw_ds_main = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT,streaming=False)
    def label_gender(example):
      if example["gender"] == "Male":
          example["SPEAKER_COLUMN"] = "Raman"
      else:
          example["SPEAKER_COLUMN"] = "Anvita"
      return example

    # Apply the transformation to create or overwrite a column
    raw_ds_main = raw_ds_main.map(label_gender)
    total_split = 2
    splits = np.linspace(0, len(raw_ds_main), num=total_split)
    splits.astype(int)
    for i in range(splits.size -1):
      raw_ds = raw_ds_main.select(range(int(splits[i]),int(splits[i+1]-1)))
      raw_ds = raw_ds.cast_column(AUDIO_COLUMN, Audio(sampling_rate=TARGET_SR))
      # few_rows = list(dataset.take(100))
      # raw_ds = dataset
      # Auto-detect the transcript column if not explicitly set
      ###---------enable when you are not sure about --------columns
      if TEXT_COLUMN is None:
          for cand in ["text_normalized", "text", "transcription", "transcript", "normalized_text", "sentence"]:
              if cand in raw_ds.column_names:
                  TEXT_COLUMN = cand
                  break
          if TEXT_COLUMN is None:
              raise ValueError(f"Could not find a text column in {raw_ds.column_names}. Set TEXT_COLUMN manually.")

      # Auto-detect the speaker column (multi-speaker conditioning)
      if SPEAKER_COLUMN is None:
          for cand in ["speaker_id", "speaker", "speaker_name", "client_id", "speaker_idx","SPEAKER_COLUMN"]:
              if cand in raw_ds.column_names:
                  SPEAKER_COLUMN = cand
                  break
          if SPEAKER_COLUMN is None:
              raise ValueError(f"Could not find a speaker column in {raw_ds.column_names}. Set SPEAKER_COLUMN manually.")
      print(f">>> Columns -> audio='{AUDIO_COLUMN}', text='{TEXT_COLUMN}', speaker='{SPEAKER_COLUMN}'.")
      print(f">>> Distinct speakers in split: {len(set(raw_ds[SPEAKER_COLUMN]))}")
      ###---------enable when you are not sure about --------columns
      get_gpu_memory_map()
      # Decode + resample audio to 24kHz on access
      raw_ds = raw_ds.cast_column(AUDIO_COLUMN, Audio(sampling_rate=TARGET_SR))

      # Drop clips longer than MAX_AUDIO_SECONDS so we don't truncate speech mid-utterance
      def _within_duration(example):
          audio = example[AUDIO_COLUMN]
          return len(audio["array"]) / audio["sampling_rate"] <= MAX_AUDIO_SECONDS

      print(">>> Filtering overly long clips...")
      raw_ds = raw_ds.filter(_within_duration)

      # # Sorting by clip length keeps each batch's padding minimal -> faster + less waste.
      # print(">>> Sorting by duration to minimize padding within batches...")
      raw_ds = raw_ds.map(lambda ex: {"_nsamp": len(ex[AUDIO_COLUMN]["array"])},num_proc=3)
      raw_ds = raw_ds.sort("_nsamp").remove_columns("_nsamp")

      # NOTE: SNAC runs on the GPU, so keep num_proc=1 (single CUDA worker). Batching is
      # what makes this fast -- the whole batch is encoded in one forward pass.
      print(">>> Batch-encoding audio -> SNAC tokens and tokenizing (this can take a while)...")
      dataset = raw_ds.map(
          encode_batch,
          batched=True,
          batch_size=32,
          remove_columns=raw_ds.column_names,
          # num_proc=3,
          writer_batch_size=100,
      )

      print(f">>> Caching encoded dataset to '{ENCODED_CACHE_DIR}' for instant reuse...")
      dataset.save_to_disk(ENCODED_CACHE_DIR+f"part_{i}")
      get_gpu_memory_map()

print(f">>> Prepared {len(dataset)} training examples.")


>>> SNAC compression: ~2000.0 samples per coarse frame
>>> Loading dataset 'ai4bharat/Rasa' (split='train')...


Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/26 [00:00<?, ?it/s]

>>> Columns -> audio='audio', text='text', speaker='SPEAKER_COLUMN'.
>>> Distinct speakers in split: 2
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
>>> Filtering overly long clips...


...

>>> Batch-encoding audio -> SNAC tokens and tokenizing (this can take a while)...


Map:   0%|          | 0/25710 [00:00<?, ? examples/s]

Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
Clearninng up Memory
Allocated: 1248.42 

Saving the dataset (0/1 shards):   0%|          | 0/25710 [00:00<?, ? examples/s]

Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1336.00 MB
>>> Prepared 25710 training examples.


In [29]:
dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 31710
})

In [30]:
import torch
import gc

def get_gpu_memory_map():
    print(f"Clearninng up Memory")
    # 1. Clear Python references and garbage collect
    gc.collect()

    # 2. Flush the PyTorch CUDA memory allocator cache
    torch.cuda.empty_cache()

    # 3. Reset peak memory statistics (optional but helpful)
    torch.cuda.reset_peak_memory_stats()

    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

get_gpu_memory_map()

Clearninng up Memory
Allocated: 1248.42 MB
Reserved: 1340.00 MB


In [ ]:
d

In [ ]:
# !rm -rf "/content/qwen_orpheus_snac_output3"

In [ ]:

from datasets import load_from_disk
from datasets import concatenate_datasets
print(f">>> Found cached encoded dataset at '{ENCODED_CACHE_DIR}', loading it...")
dataset1 = load_from_disk("/content/qwen-snac-tts/tts_encoded_cache")
dataset2 = load_from_disk("/content/qwen-snac-tts/tts_encoded_cache_others_500_27k")
dataset = concatenate_datasets([dataset1,dataset2])
print(f">>> Prepared {len(dataset)} training examples.")


>>> Found cached encoded dataset at '/content/qwen-snac-tts/tts_encoded_cache_360', loading it...
>>> Prepared 59512 training examples.


In [ ]:
#=========================================
# 6. EXECUTE HUGGINGFACE TRAINER RUN
# ==========================================
print(">>> Setting up training arguments...")
trainer = Trainer(
    model=model,
    # processing_class=tokenizer,
    tokenizer=tokenizer,
    train_dataset=dataset,
    # We pre-tokenized manually above, so a plain DataCollator is all we need.
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
    args=TrainingArguments(
        # per_device_train_batch_size=128,
        # gradient_accumulation_steps=2,
               per_device_train_batch_size=4,
        # Change from 2 to 8 to keep total batch size at 256 (32 x 8 x 1 = 256)
        gradient_accumulation_steps=64,

        group_by_length=True, # Dramatically reduces padding overhead
        warmup_steps=60,
        num_train_epochs=3,        # full passes over the dataset (set max_steps=-1 to honor this)
        max_steps=-1,
        # learning_rate=2e-5,
        # learning_rate=1.2e-4,
        learning_rate=1.2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,

        output_dir="qwen_orpheus_snac_output3",
        weight_decay=0.06,
        # lr_scheduler_type="cosine",
        lr_scheduler_type="cosine_with_restarts",
        optim="adamw_torch_fused",
        seed=3407,
        report_to = "none", # Use TrackIO/WandB etc
        max_grad_norm=0.5,


    ),
)

print(">>> Starting optimization pass via Unsloth...")
trainer_stats = trainer.train(resume_from_checkpoint=False)
print(">>> Training finalized successfully.")

# ==========================================
# 7. EXPORT MODEL CONFIGURATION
# ==========================================
print(">>> Exporting LoRA adapter + extended tokenizer...")
# Saves your text-to-speech adapter (incl. trained embed_tokens/lm_head) and tokenizer
# model.save_pretrained_merged("qwen_snac_tts_final", tokenizer, save_method="lora")
print(">>> Setup Complete. Check outputs in 'qwen_snac_tts_final' folder.")

>>> Setting up training arguments...
>>> Starting optimization pass via Unsloth...


/usr/local/lib/python3.12/dist-packages/unsloth/models/_utils.py:2568: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  _original_trainer_init(self, *args, **kwargs)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25,710 | Num Epochs = 3 | Total steps = 303
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 64
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 64 x 1) = 256
 "-____-"     Trainable parameters = 608,362,496 of 608,362,496 (100.00% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,3.148600
20,2.917200
30,2.814900
40,2.718600
50,2.582400
60,2.695300


In [ ]:
print(">>> Exporting LoRA adapter + extended tokenizer...")
# Saves your text-to-speech adapter (incl. trained embed_tokens/lm_head) and tokenizer
# model.save_pretrained_merged("qwen_snac_tts_final_lora", tokenizer, save_method="lora")
model.save_pretrained_merged("qwen_snac_tts_final_16b", tokenizer, save_method="merged_16bit")
print(">>> Setup Complete. Check outputs in 'qwen_snac_tts_final' folder.")

In [ ]:
import torch
import torchaudio
import re
from unsloth import FastLanguageModel
from snac import SNAC
from google.colab import userdata
HF_TOKEN =userdata.get('HF_TOKEN')


# from huggingface_hub import snapshot_download
# snapshot_download("Nikya/qupus_tts_0.8Bs",token=HF_TOKEN,local_dir="qupus_tts_0.8Bs")
# ==========================================
# 1. INITIALIZATION & CONFIGURATION
# ==========================================
# Point this path to the output directory generated by your training script
TRAINED_MODEL_PATH = "qupus_tts_0.8Bs"
MAX_GEN_TOKENS = 1024  # Max audio tokens to generate (7 tokens = 1 SNAC frame)
DTYPE = torch.bfloat16 # Match training precision

# Pick any speaker id/name that appeared in training (LibriTTS uses numeric ids).
DEFAULT_SPEAKER = None

# print(">>> Loading extended tokenizer and trained Qwen model weights...")
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=TRAINED_MODEL_PATH,
#     max_seq_length=2048,
#     dtype=DTYPE,
#     load_in_4bit=True, # Keeps inference highly optimized on lower VRAM footprints
# )
# FastLanguageModel.for_inference(model) # Drastically optimizes generation speed

print(">>> Loading 24kHz SNAC decoder architecture...")
snac_decoder = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().cuda()

# Resolve the stop token id exactly (encode() can prepend specials -> wrong id)
AUDIO_END_ID = tokenizer.convert_tokens_to_ids("<audio_end>")

# Expected layer pattern of one 7-token SNAC frame (must match training interleave)
FRAME_LAYER_PATTERN = [0, 1, 2, 2, 1, 2, 2]

def build_prompt_prefix(speaker, text_prompt):
    """MUST match the training prefix in the training cell exactly."""
    return f"{speaker}: {text_prompt} <audio_start> "

# ==========================================
# 2. INFERENCE ENGINE EXECUTION
# ==========================================
# def generate_speech(text_to_synthesize, speaker=DEFAULT_SPEAKER, output_filename="output_generation.wav"):
#     """
#     Feeds the speaker-conditioned text prompt into Qwen, streams the custom token
#     outputs, reconstructs the 3-layer SNAC matrix, and exports a clean audio file.
#     """
#     # Build prompt template matching the training data formatting EXACTLY.
#     # Training tokenized the prefix with add_special_tokens=False, so do the same.
#     prompt_input = build_prompt_prefix(speaker, text_to_synthesize)

#     print(f"\n>>> Encoding prompt (speaker='{speaker}'): '{text_to_synthesize}'")
#     input_ids = tokenizer(prompt_input, add_special_tokens=False, return_tensors="pt").input_ids.cuda()

#     print(">>> Generating audio token stream from Qwen backbone...")
#     with torch.inference_mode():
#         output_ids = model.generate(
#             input_ids=input_ids,
#             max_new_tokens=MAX_GEN_TOKENS,
#             use_cache=True,
#              do_sample = True,
#           temperature = 1.0,
#             top_p = 0.95,
#             repetition_penalty = 1.1,
#             num_return_sequences = 1,
#             eos_token_id=AUDIO_END_ID, # Stop when audio boundary ends
#         )

#     # Extract only the newly generated tokens (slice out the input prompt)
#     generated_tokens_ids = output_ids[0][len(input_ids[0]):]

#     # Map ids -> raw token strings directly (no decode() spacing/normalization quirks)
#     decoded_string_tokens = tokenizer.convert_ids_to_tokens(generated_tokens_ids.tolist())

#     # Collect (layer, code) pairs in generation order
#     token_pattern = re.compile(r"<snac_l(\d+)_c(\d+)>")
#     parsed = []
#     for tok in decoded_string_tokens:
#         match = token_pattern.fullmatch(tok)
#         if match:
#             parsed.append((int(match.group(1)), int(match.group(2))))

#     print(">>> Re-assembling sequential audio tokens into hierarchical grids...")
#     layer0_codes, layer1_codes, layer2_codes = [], [], []

#     # Read 7 tokens per frame and de-interleave. Stop at the first malformed frame.
#     n_frames = len(parsed) // 7
#     valid_frames = 0
#     for f in range(n_frames):
#         frame = parsed[f * 7:(f + 1) * 7]
#         if [layer for layer, _ in frame] != FRAME_LAYER_PATTERN:
#             break  # generation drifted out of the expected layer order
#         codes = [c for _, c in frame]
#         layer0_codes.append(codes[0])
#         layer1_codes.extend([codes[1], codes[4]])
#         layer2_codes.extend([codes[2], codes[3], codes[5], codes[6]])
#         valid_frames += 1

#     if valid_frames == 0:
#         print("❌ Error: The model failed to generate a coherent chain of audio frames.")
#         return

#     print(f">>> Processing {valid_frames} synchronized audio frames "
#           f"({len(layer0_codes)}/{len(layer1_codes)}/{len(layer2_codes)} codes per layer)...")

#     # Tensors structured for SNAC input: each is [batch=1, time_steps]
#     l0_tensor = torch.tensor([layer0_codes], dtype=torch.long).cuda()
#     l1_tensor = torch.tensor([layer1_codes], dtype=torch.long).cuda()
#     l2_tensor = torch.tensor([layer2_codes], dtype=torch.long).cuda()
#     snac_input_codes = [l0_tensor, l1_tensor, l2_tensor]

#     print(">>> Running SNAC neural decoder pass...")
#     with torch.inference_mode():
#         reconstructed_waveform = snac_decoder.decode(snac_input_codes)

#     # Move to CPU, shape [1, samples] for saving at SNAC's native 24kHz
#     final_audio = reconstructed_waveform.squeeze(0).cpu()
#     torchaudio.save(output_filename, final_audio, sample_rate=24000)
#     print(f"✅ Success! Audio file saved directly to: '{output_filename}'")
def generate_speech(text_to_synthesize, speaker=DEFAULT_SPEAKER, output_filename="qwen_orpheus_test.wav"):
    """
    Feeds speaker-conditioned text into Qwen, enforces tight sampling boundaries,
    safely de-interleaves tokens, and exports a functional 24kHz audio waveform.
    """
    prompt_input = build_prompt_prefix(speaker, text_to_synthesize)

    print(f"\n>>> Encoding prompt (speaker='{speaker}'): '{text_to_synthesize}'")
    input_ids = tokenizer(prompt_input, add_special_tokens=False, return_tensors="pt").input_ids.cuda()

    print(">>> Generating audio token stream from Qwen backbone...")
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_GEN_TOKENS,
            use_cache=True,

            # --- CRITICAL SAMPLING CORRECTIONS ---
            do_sample=True,
            temperature=0.3,         # Lower temperature forces structural pattern compliance
            # top_k=100,                # Clamp options strictly to correct codebook values
            top_p=0.95,              # Exclude low-probability acoustic glitch tokens
            repetition_penalty=1.1,  # Keep this to avoid repeating single frames infinitely

            eos_token_id=AUDIO_END_ID,
        )

    # Extract only the newly generated tokens
    generated_tokens_ids = output_ids[0][len(input_ids[0]):]

    # Convert ids back to structural string tokens
    decoded_string_tokens = tokenizer.convert_ids_to_tokens(generated_tokens_ids.tolist())

    # Collect layer and code indices matching your exact pattern format
    token_pattern = re.compile(r"<snac_l(\d+)_c(\d+)>")
    parsed = []
    for tok in decoded_string_tokens:
        match = token_pattern.fullmatch(tok)
        if match:
            parsed.append((int(match.group(1)), int(match.group(2))))

    print(f">>> Extracted {len(parsed)} structural SNAC string tokens.")

    # Initialize structural arrays
    layer0_codes, layer1_codes, layer2_codes = [], [], []
    valid_frames = 0

    # Process using a rolling window to match frames even if a token drops out
    i = 0
    while i <= len(parsed) - 7:
        frame = parsed[i:i+7]
        current_pattern = [layer for layer, _ in frame]

        # Check if this 7-token block matches your strict frame design
        if current_pattern == FRAME_LAYER_PATTERN:
            codes = [c for _, c in frame]
            layer0_codes.append(codes[0])
            layer1_codes.extend([codes[1], codes[4]])
            layer2_codes.extend([codes[2], codes[3], codes[5], codes[6]])
            valid_frames += 1
            i += 7 # Advance by a full valid frame block
        else:
            # If mismatched, advance by 1 token to find the next structural boundary alignment
            i += 1

    if valid_frames == 0:
        print("❌ Error: Zero synchronized audio frames could be extracted from this sequence.")
        print(f"Sample of raw tokens parsed: {parsed[:14]}")
        return

    print(f">>> Successfully parsed {valid_frames} clean audio frames.")
    print(f"Layer lengths -> L0: {len(layer0_codes)}, L1: {len(layer1_codes)}, L2: {len(layer2_codes)}")

    # Format multi-layer arrays into individual parallel tensors
    l0_tensor = torch.tensor([layer0_codes], dtype=torch.long).cuda()
    l1_tensor = torch.tensor([layer1_codes], dtype=torch.long).cuda()
    l2_tensor = torch.tensor([layer2_codes], dtype=torch.long).cuda()
    snac_input_codes = [l0_tensor, l1_tensor, l2_tensor]

    print(">>> Executing SNAC neural decoder pass...")
    with torch.inference_mode():
        reconstructed_waveform = snac_decoder.decode(snac_input_codes)

    # Reshape and export waveform to disk
    final_audio = reconstructed_waveform.squeeze(0).cpu()
    torchaudio.save(output_filename, final_audio, sample_rate=24000)
    print(f"✅ Success! Generated audio file saved to: '{output_filename}'")
    from IPython.display import Audio

    display(Audio("qwen_orpheus_test.wav"))

# ==========================================
# 3. RUNTIME TEST UTILITY
# ==========================================
if __name__ == "__main__":
    test_phrase = "Hello what is your name?"
    # test_phrase = "Already my slow steps had carried me Into the ancient wood so far, that I Could not perceive where I had entered it."
    # Pass speaker= to switch voices (must be an id/name seen during training).
    generate_speech(test_phrase, speaker=DEFAULT_SPEAKER, output_filename="qwen_orpheus_test.wav")

>>> Loading 24kHz SNAC decoder architecture...

>>> Encoding prompt (speaker='None'): 'Hello what is your name?'
>>> Generating audio token stream from Qwen backbone...
>>> Extracted 161 structural SNAC string tokens.
>>> Successfully parsed 23 clean audio frames.
Layer lengths -> L0: 23, L1: 46, L2: 92
>>> Executing SNAC neural decoder pass...
✅ Success! Generated audio file saved to: 'qwen_orpheus_test.wav'


In [ ]:
test_phrase = "Hi! this is qwen text to speech model, developed by NIKHIL"
# test_phrase = "Already my slow steps had carried me Into the ancient wood so far, that I Could not perceive where I had entered it."
# Pass speaker= to switch voices (must be an id/name seen during training).
generate_speech(test_phrase, speaker=DEFAULT_SPEAKER, output_filename="qwen_orpheus_test.wav")


>>> Encoding prompt (speaker='None'): 'Hi! this is qwen text to speech model, developed by NIKHIL'
>>> Generating audio token stream from Qwen backbone...
>>> Extracted 441 structural SNAC string tokens.
>>> Successfully parsed 63 clean audio frames.
Layer lengths -> L0: 63, L1: 126, L2: 252
>>> Executing SNAC neural decoder pass...
✅ Success! Generated audio file saved to: 'qwen_orpheus_test.wav'


In [ ]:
from IPython.display import Audio


# Make sure the file name matches your saved audio file path exactly
display(Audio("qwen_orpheus_test.wav"))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
m = AutoModelForCausalLM.from_pretrained("Nikya/qupus_tts_0.8Bs", torch_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained("Nikya/qupus_tts_0.8Bs")
print("tie_word_embeddings:", m.config.tie_word_embeddings)
print("vocab/emb/lmhead:", len(tok),
      m.get_input_embeddings().weight.shape[0],
      m.get_output_embeddings().weight.shape[0])
ids = [tok.convert_tokens_to_ids(f"<snac_l0_c{c}>") for c in range(20)]
emb, lm = m.get_input_embeddings().weight, m.get_output_embeddings().weight
print("emb new-tok mean-abs :", emb[ids].abs().mean().item())
print("lmhead new-tok mean-abs:", lm[ids].abs().mean().item())
print("emb == lmhead?", torch.allclose(emb, lm))


In [ ]:
import torch
import torchaudio
import re
from google.colab import userdata
HF_TOKEN =userdata.get('HF_TOKEN')
# NOTE: do NOT import unsloth here. Importing it monkey-patches transformers'
# attention (expecting `apply_qkv`), which crashes a vanilla AutoModel load.
# This cell tests the model exactly as a deployed HF user would -- pure transformers.
from transformers import AutoTokenizer, AutoModelForCausalLM
from snac import SNAC

# ==========================================
# 1. INITIALIZATION & CONFIGURATION
# ==========================================
# Point this at your saved/standalone checkpoint (full-FT output or merged model).
TRAINED_MODEL_PATH = "/content/qupus_tts_0.8B"
MAX_GEN_TOKENS = 1024  # Max audio tokens to generate (7 tokens = 1 SNAC frame)
DTYPE = torch.bfloat16 # Match training precision (bf16); float16 can overflow

# Pick any speaker id/name that appeared in training (LibriTTS uses numeric ids).
# IMPORTANT: this must be a speaker the model actually saw, or output will be junk.
DEFAULT_SPEAKER = "3081"

print(">>> Loading trained model with vanilla transformers (no unsloth)...")
tokenizer = AutoTokenizer.from_pretrained(TRAINED_MODEL_PATH,token = HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    TRAINED_MODEL_PATH,
    device_map="cuda",
    torch_dtype=DTYPE,
    token = HF_TOKEN
)
model.eval()

print(">>> Loading 24kHz SNAC decoder architecture...")
snac_decoder = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().cuda()

# Resolve the stop token id exactly (encode() can prepend specials -> wrong id)
AUDIO_END_ID = tokenizer.convert_tokens_to_ids("<audio_end>")

# Expected layer pattern of one 7-token SNAC frame (must match training interleave)
FRAME_LAYER_PATTERN = [0, 1, 2, 2, 1, 2, 2]

def build_prompt_prefix(speaker, text_prompt):
    """MUST match the training prefix in the training cell exactly."""
    return f"{speaker}: {text_prompt} <audio_start> "

def deinterleave_with_resync(parsed):
    """Rebuild the 3 SNAC layers from a flat (layer, code) stream.

    Instead of bailing on the first bad frame, resync: only accept a 7-token window
    that starts at layer 0 and matches FRAME_LAYER_PATTERN; otherwise skip one token
    and try again. This recovers all clean frames even if the stream has glitches.
    """
    l0, l1, l2 = [], [], []
    i, n, kept, skipped = 0, len(parsed), 0, 0
    while i + 7 <= n:
        window = parsed[i:i + 7]
        if [layer for layer, _ in window] == FRAME_LAYER_PATTERN:
            codes = [c for _, c in window]
            l0.append(codes[0])
            l1.extend([codes[1], codes[4]])
            l2.extend([codes[2], codes[3], codes[5], codes[6]])
            kept += 1
            i += 7
        else:
            skipped += 1
            i += 1  # resync to the next possible frame boundary
    return l0, l1, l2, kept, skipped

# ==========================================
# 2. INFERENCE ENGINE EXECUTION
# ==========================================
def generate_speech(text_to_synthesize, speaker=DEFAULT_SPEAKER, output_filename="output_generation.wav",
                    temperature=0.7, top_p=0.9, debug=True):
    """
    Feeds the speaker-conditioned text prompt into Qwen, streams the custom token
    outputs, reconstructs the 3-layer SNAC matrix, and exports a clean audio file.
    """
    # Build prompt template matching the training data formatting EXACTLY.
    prompt_input = build_prompt_prefix(speaker, text_to_synthesize)

    print(f"\n>>> Encoding prompt (speaker='{speaker}'): '{text_to_synthesize}'")
    input_ids = tokenizer(prompt_input, add_special_tokens=False, return_tensors="pt").input_ids.to(model.device)

    print(">>> Generating audio token stream from Qwen backbone...")
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_GEN_TOKENS,
            use_cache=True,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=1.1,   # discourages the model getting stuck on one token
            eos_token_id=AUDIO_END_ID,
        )

    # Extract only the newly generated tokens (slice out the input prompt)
    generated_tokens_ids = output_ids[0][len(input_ids[0]):]
    decoded_string_tokens = tokenizer.convert_ids_to_tokens(generated_tokens_ids.tolist())

    # Collect (layer, code) pairs in generation order
    token_pattern = re.compile(r"<snac_l(\d+)_c(\d+)>")
    parsed = []
    for tok in decoded_string_tokens:
        m = token_pattern.fullmatch(tok)
        if m:
            parsed.append((int(m.group(1)), int(m.group(2))))

    # ---- Diagnostics: what did the model actually produce? ----
    if debug:
        n_gen = len(decoded_string_tokens)
        n_snac = len(parsed)
        print(f">>> Generated {n_gen} tokens; {n_snac} are SNAC audio tokens "
              f"({100 * n_snac / max(n_gen, 1):.0f}%).")
        print(f">>> First 14 generated tokens: {decoded_string_tokens[:14]}")
        if n_snac:
            print(f">>> First frame layers: {[layer for layer, _ in parsed[:7]]} "
                  f"(expected {FRAME_LAYER_PATTERN})")

    print(">>> Re-assembling sequential audio tokens into hierarchical grids...")
    layer0_codes, layer1_codes, layer2_codes, kept, skipped = deinterleave_with_resync(parsed)

    if kept == 0:
        print("❌ Error: no valid SNAC frames recovered.")
        print("    Likely causes:")
        print("      - model emitted few/no <snac_*> tokens  -> undertrained, or prompt format mismatch")
        print("      - speaker id not seen in training        -> set DEFAULT_SPEAKER to a real one")
        print("      - checkpoint loaded incorrectly          -> verify TRAINED_MODEL_PATH")
        return

    if skipped:
        print(f">>> Recovered {kept} frames, resynced past {skipped} stray/malformed tokens.")

    print(f">>> Processing {kept} synchronized audio frames "
          f"({len(layer0_codes)}/{len(layer1_codes)}/{len(layer2_codes)} codes per layer)...")

    # Tensors structured for SNAC input: each is [batch=1, time_steps]
    l0_tensor = torch.tensor([layer0_codes], dtype=torch.long).cuda()
    l1_tensor = torch.tensor([layer1_codes], dtype=torch.long).cuda()
    l2_tensor = torch.tensor([layer2_codes], dtype=torch.long).cuda()
    snac_input_codes = [l0_tensor, l1_tensor, l2_tensor]

    print(">>> Running SNAC neural decoder pass...")
    with torch.inference_mode():
        reconstructed_waveform = snac_decoder.decode(snac_input_codes)

    # Move to CPU, shape [1, samples] for saving at SNAC's native 24kHz
    final_audio = reconstructed_waveform.squeeze(0).cpu()
    torchaudio.save(output_filename, final_audio, sample_rate=24000)
    print(f"✅ Success! Audio file saved directly to: '{output_filename}'")

# ==========================================
# 3. RUNTIME TEST UTILITY
# ==========================================
if __name__ == "__main__":
    test_phrase = "Hello from the newly upgraded Qwen and SNAC text to speech engine. It works flawlessly!"
    # Pass speaker= to switch voices (must be an id/name seen during training).
    generate_speech(test_phrase, speaker=DEFAULT_SPEAKER, output_filename="qwen_orpheus_test.wav")

>>> Loading trained model with vanilla transformers (no unsloth)...
>>> Loading 24kHz SNAC decoder architecture...

>>> Encoding prompt (speaker='3081'): 'Hello from the newly upgraded Qwen and SNAC text to speech engine. It works flawlessly!'
>>> Generating audio token stream from Qwen backbone...
>>> Generated 1024 tokens; 512 are SNAC audio tokens (50%).
>>> First 14 generated tokens: ['<snac_l0_c1036>', 'Ġ', '<snac_l1_c4049>', 'Ġ', '<snac_l2_c1211>', 'Ġ', '<snac_l2_c355>', 'Ġ', '<snac_l1_c2617>', 'Ġ', '<snac_l2_c2573>', 'Ġ', '<snac_l2_c897>', 'Ġ']
>>> First frame layers: [0, 1, 2, 2, 1, 2, 2] (expected [0, 1, 2, 2, 1, 2, 2])
>>> Re-assembling sequential audio tokens into hierarchical grids...
>>> Processing 73 synchronized audio frames (73/146/292 codes per layer)...
>>> Running SNAC neural decoder pass...
✅ Success! Audio file saved directly to: 'qwen_orpheus_test.wav'


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "./qupus_tts_0.8Bs"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

# 1. Print all special/added tokens to see if SNAC is registered
print("--- Added Tokens Check ---")
for token, token_id in tokenizer.get_added_vocab().items():
    if "snac" in token.lower() or "speaker" in token.lower():
        print(f"Token: {token} -> ID: {token_id}")

# 2. Check if the model's output layer size matches the tokenizer
print("\n--- Structural Alignment ---")
print(f"Tokenizer Size: {len(tokenizer)}")
print(f"Model Output Layer (lm_head) Size: {model.lm_head.out_features}")


In [ ]:
!rm -rf "/content/qupus_tts_0.8B"

In [ ]:
# ==========================================
# 8. EXPORT A STANDALONE HUGGINGFACE MODEL
# ==========================================
# The cell above saved only the LoRA *adapter* (loadable via Unsloth/PEFT).
# To get a plain `transformers` model that loads WITHOUT unsloth, merge the
# adapter into the base weights. Because we resized the vocab and trained
# embed_tokens/lm_head, "merged_16bit" is required so the enlarged embedding
# matrix + lm_head (and the new SNAC tokens) are baked into the checkpoint.
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_OUTPUT_DIR = "qupus_tts_0.8B2"

print(">>> Merging LoRA into base weights and saving a standalone fp16 checkpoint...")
model.save_pretrained_merged(HF_OUTPUT_DIR, tokenizer, save_method="merged_16bit")
print(f">>> Saved standard transformers model to '{HF_OUTPUT_DIR}'.")

# ---- Sanity check: reload with vanilla transformers (no unsloth in the loop) ----
print(">>> Verifying it loads with plain AutoModelForCausalLM...")
hf_tokenizer = AutoTokenizer.from_pretrained(HF_OUTPUT_DIR)
hf_model = AutoModelForCausalLM.from_pretrained(
    HF_OUTPUT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
assert len(hf_tokenizer) == hf_model.get_input_embeddings().weight.shape[0], \
    "Tokenizer vocab size and embedding rows disagree -- the resize did not persist."
print(f">>> OK. Vocab size = {len(hf_tokenizer)} (base + 2 control + 12,288 SNAC tokens).")

# ---- (Optional) Push to the HuggingFace Hub ----
# Requires `huggingface-cli login` or a token. Uncomment to publish.
HF_REPO = "Nikya/qupus_tts_0.8Bsbase"
model.push_to_hub_merged(HF_REPO, tokenizer, token=HF_TOKEN,save_method='merged_16bit')
# tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f">>> Pushed merged model to https://huggingface.co/{HF_REPO}")

# ---- (Optional) GGUF export for llama.cpp / Ollama inference ----
# model.save_pretrained_gguf(HF_OUTPUT_DIR + "_gguf", tokenizer, quantization_method="q8_0")

>>> Merging LoRA into base weights and saving a standalone fp16 checkpoint...
Unsloth: Saving full fine-tuned model to 'qupus_tts_0.8B2' ...
Unsloth: Model saved successfully to 'qupus_tts_0.8B2'
>>> Saved standard transformers model to 'qupus_tts_0.8B2'.
>>> Verifying it loads with plain AutoModelForCausalLM...
>>> OK. Vocab size = 163960 (base + 2 control + 12,288 SNAC tokens).
Unsloth: Pushing full fine-tuned model to 'Nikya/qupus_tts_0.8Bsbase' ...


README.md:   0%|          | 0.00/529 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...8Bsbase/model.safetensors:   0%|          |  613kB / 1.22GB            

Saved model to https://huggingface.co/Nikya/qupus_tts_0.8Bsbase


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._0.8Bsbase/tokenizer.json: 100%|##########| 13.8MB / 13.8MB            

Unsloth: Model saved successfully to 'Nikya/qupus_tts_0.8Bsbase'
>>> Pushed merged model to https://huggingface.co/Nikya/qupus_tts_0.8Bsbase


In [ ]:
from google.colab import runtime
runtime.unassign()

## Uploading prepared data



In [33]:
from huggingface_hub import HfApi, create_repo
import os

# Replace with your desired HuggingFace username and repository name
HF_USERNAME = "Nikya" # e.g., "unsloth"
# HF_REPO_NAME = "qwen-snac-tts-0.8B" # e.g., "qwen-snac-tts-merged"
HF_REPO_NAME = "qwen-snac-tts-finetuning" # e.g., "qwen-snac-tts-merged"

# Your HuggingFace token (you can also run `huggingface-cli login` in your terminal)
# If you have your token stored as a Colab Secret, you can access it like this:
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')
from google.colab import userdata
HF_TOKEN =userdata.get('HF_TOKEN')


# Initialize the HfApi
api = HfApi(token=HF_TOKEN)

# Create the repository if it doesn't exist
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
print(f"Checking for repository: {repo_id}")

try:
    create_repo(repo_id=repo_id, token=HF_TOKEN, private=True, exist_ok=True)
    print(f"Repository '{repo_id}' ensured to exist.")
except Exception as e:
    print(f"Could not create or verify repo: {e}")
    print("Please ensure you have configured your HuggingFace token correctly and have push permissions.")


LOCAL_MODEL_DIR = "/content/tts_encoded_cache_hindipart_0" # Defined as 'qwen_snac_tts_merged' in a previous cell
TARGET_FOLDER_IN_REPO="tts_encoded_cache_hindi"
# Upload the contents of the local directory to the repository
print(f"Uploading content from '{LOCAL_MODEL_DIR}' to '{repo_id}'...")

# To upload a folder, you can use the upload_folder method.
# The path_in_repo argument allows you to specify a subfolder in the repository
# If not specified, files are uploaded to the root of the repo.
api.upload_folder(
    folder_path=LOCAL_MODEL_DIR,
    repo_id=repo_id,
    repo_type="model", # Can be 'model', 'dataset', or 'space'
     path_in_repo=TARGET_FOLDER_IN_REPO,
    commit_message=f"Upload {LOCAL_MODEL_DIR} folder",
    token=HF_TOKEN,
)

print(f"Successfully uploaded '{LOCAL_MODEL_DIR}' to https://huggingface.co/{repo_id}")


# from google.colab import runtime
# runtime.unassign()

Checking for repository: Nikya/qwen-snac-tts-finetuning
Repository 'Nikya/qwen-snac-tts-finetuning' ensured to exist.
Uploading content from '/content/tts_encoded_cache_hindipart_0' to 'Nikya/qwen-snac-tts-finetuning'...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...data-00000-of-00001.arrow:   0%|          | 1.77MB /  359MB            

Successfully uploaded '/content/tts_encoded_cache_hindipart_0' to https://huggingface.co/Nikya/qwen-snac-tts-finetuning


### Uploading to HuggingFace Hub

To upload your model to the HuggingFace Hub, you'll need to:
1. Log in to HuggingFace CLI or set your token programmatically.
2. Define your desired repository name.
3. Use `huggingface_hub` to push the folder.

In [ ]:
from huggingface_hub import HfApi, create_repo
import os

# Replace with your desired HuggingFace username and repository name
HF_USERNAME = "Nikya" # e.g., "unsloth"
# HF_REPO_NAME = "qwen-snac-tts-0.8B" # e.g., "qwen-snac-tts-merged"
HF_REPO_NAME = "qwen-snac-tts-finetuning" # e.g., "qwen-snac-tts-merged"

# Your HuggingFace token (you can also run `huggingface-cli login` in your terminal)
# If you have your token stored as a Colab Secret, you can access it like this:
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')
from google.colab import userdata
HF_TOKEN =userdata.get('HF_TOKEN')


# Initialize the HfApi
api = HfApi(token=HF_TOKEN)

# Create the repository if it doesn't exist
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
print(f"Checking for repository: {repo_id}")

try:
    create_repo(repo_id=repo_id, token=HF_TOKEN, private=True, exist_ok=True)
    print(f"Repository '{repo_id}' ensured to exist.")
except Exception as e:
    print(f"Could not create or verify repo: {e}")
    print("Please ensure you have configured your HuggingFace token correctly and have push permissions.")

# Define the local directory to upload (this is the merged model output from the previous step)
LOCAL_MODEL_DIR = "/content/qupus_tts_0.8B" # Defined as 'qwen_snac_tts_merged' in a previous cell
TARGET_FOLDER_IN_REPO="qupus-model"
# Upload the contents of the local directory to the repository
print(f"Uploading content from '{LOCAL_MODEL_DIR}' to '{repo_id}'...")

# To upload a folder, you can use the upload_folder method.
# The path_in_repo argument allows you to specify a subfolder in the repository
# If not specified, files are uploaded to the root of the repo.
api.upload_folder(
    folder_path=LOCAL_MODEL_DIR,
    repo_id=repo_id,
    repo_type="model", # Can be 'model', 'dataset', or 'space'
     path_in_repo=TARGET_FOLDER_IN_REPO,
    commit_message=f"Upload {LOCAL_MODEL_DIR} folder",
    token=HF_TOKEN,
)

print(f"Successfully uploaded '{LOCAL_MODEL_DIR}' to https://huggingface.co/{repo_id}")


LOCAL_MODEL_DIR = "/content/qwen_orpheus_snac_output2/checkpoint-2100" # Defined as 'qwen_snac_tts_merged' in a previous cell
TARGET_FOLDER_IN_REPO="qupus-checkpoint/checkpoint-2100"
# Upload the contents of the local directory to the repository
print(f"Uploading content from '{LOCAL_MODEL_DIR}' to '{repo_id}'...")

# To upload a folder, you can use the upload_folder method.
# The path_in_repo argument allows you to specify a subfolder in the repository
# If not specified, files are uploaded to the root of the repo.
api.upload_folder(
    folder_path=LOCAL_MODEL_DIR,
    repo_id=repo_id,
    repo_type="model", # Can be 'model', 'dataset', or 'space'
     path_in_repo=TARGET_FOLDER_IN_REPO,
    commit_message=f"Upload {LOCAL_MODEL_DIR} folder",
    token=HF_TOKEN,
)

print(f"Successfully uploaded '{LOCAL_MODEL_DIR}' to https://huggingface.co/{repo_id}")

Checking for repository: Nikya/qwen-snac-tts-finetuning
Repository 'Nikya/qwen-snac-tts-finetuning' ensured to exist.
Uploading content from '/content/qupus_tts_0.8B' to 'Nikya/qwen-snac-tts-finetuning'...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ts_0.8B/model.safetensors:   3%|3         | 39.9MB / 1.22GB            

  ...s_tts_0.8B/tokenizer.json: 100%|##########| 13.8MB / 13.8MB            

...

  ...ckpoint-2100/optimizer.pt:   0%|          |  557kB / 2.43GB            

  ...kpoint-2100/rng_state.pth:   8%|8         | 1.19kB / 14.6kB            

  ...nt-2100/training_args.bin:   8%|8         |   469B / 5.78kB            

  ...ckpoint-2100/scheduler.pt:   8%|8         |   119B / 1.47kB            

Successfully uploaded '/content/qwen_orpheus_snac_output2/checkpoint-2100' to https://huggingface.co/Nikya/qwen-snac-tts-finetuning
